# Brent Oil Prices — Data Preparation & EDA

Task 1 / Task 2.1: load the data, examine trend and volatility, overlay researched
key events, and test stationarity to motivate modeling log returns.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from statsmodels.tsa.stattools import adfuller

sys.path.append(str(Path.cwd().parent))
from src.data_loader import add_log_returns, load_events, load_prices

plt.rcParams["figure.figsize"] = (14, 5)

In [ ]:
prices = add_log_returns(load_prices())
events = load_events()

print(f"{len(prices)} daily observations, {prices.index.min():%Y-%m-%d} to {prices.index.max():%Y-%m-%d}")
prices.head()

## Raw price series with key events overlaid

In [ ]:
fig, ax = plt.subplots(figsize=(16, 6))
ax.plot(prices.index, prices["Price"], lw=0.8)
for _, ev in events.iterrows():
    ax.axvline(ev["event_date"], color="crimson", ls="--", lw=0.7, alpha=0.6)
ax.set_title("Brent oil price (USD/bbl) with researched key events")
ax.set_ylabel("USD per barrel")
plt.tight_layout()

## Log returns — volatility clustering

In [ ]:
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(prices.index, prices["LogReturn"], lw=0.5)
ax.set_title("Daily log returns: log(P_t) - log(P_{t-1})")
plt.tight_layout()

# Rolling 30-day std of returns — a direct view of volatility regimes
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(prices.index, prices["LogReturn"].rolling(30).std(), lw=0.8, color="darkorange")
ax.set_title("30-day rolling volatility of log returns")
plt.tight_layout()

## Stationarity: ADF tests

Expectation: the price level is non-stationary (unit root), while log returns are
stationary — which is why the change point model targets the return series.

In [ ]:
def adf_report(series, name):
    stat, pvalue, *_ = adfuller(series.dropna(), autolag="AIC")
    verdict = "stationary" if pvalue < 0.05 else "NON-stationary"
    print(f"{name:12s}  ADF stat = {stat:8.3f}   p-value = {pvalue:.4f}   -> {verdict}")

adf_report(prices["Price"], "Price")
adf_report(prices["LogReturn"], "Log returns")

## Findings (fill in after running)

- **Trend:** ...
- **Stationarity:** price level non-stationary; log returns stationary → model returns.
- **Volatility:** clustering visible around ... (compare with events overlay).
- **Modeling implication:** change points in the *mean and/or volatility of log
  returns*, or mean of price within focused windows around candidate breaks.